# 09A｜獨立測試資料的人臉裁切

將一批獨立蒐集的測試照片裁切出主要人臉，供 09C 的門檻校準使用。不進行切分、訓練或微調。

**資料來源與已知限制**

這批資料共 300 張，Real 為手機實拍照片，Fake 為網路蒐集的合成人臉影像。兩類的來源不同，除了真偽之外還系統性地存在裝置、解析度、壓縮歷程與拍攝條件的差異。

因此本組資料**不作為泛化能力的評估基準**，其 AUC 數值的上界受資料蒐集方式影響，無法排除模型部分利用了來源差異而非偽造痕跡。

它的用途是**部署可行性檢查**：驗證一個在 FF++ 上訓練的模型，其輸出機率在面對訓練分布外的真實影像時是否仍可沿用預設門檻。這個問題與 AUC 高低相對獨立——機率分布的形狀是否偏移，不因來源差異的存在而失效。

處理規則：

- 每張照片保留面積最大的主要人臉。
- 人臉框向外保留 15% 範圍後裁成正方形。
- 找不到人臉或無法讀取的圖片會跳過並記錄於 CSV。
- 輸出不預先縮放為 256×256，由後續推論階段統一處理。

資料流程：

```text
mobile_test_raw/
├── real/
└── fake/
        ↓ YOLOv11n-Face
dataset_mobile_yolo/
└── test/
    ├── real/
    └── fake/
```

## 1. 套件

In [1]:
# 缺少套件時才取消註解
# %pip install ultralytics pillow pillow-heif pandas tqdm matplotlib


In [2]:
import os
import platform
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm
from ultralytics import YOLO

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    HEIC_AVAILABLE = True
except ImportError:
    HEIC_AVAILABLE = False


def set_chinese_font():
    candidates = [
        "Microsoft JhengHei",
        "Microsoft YaHei",
        "Noto Sans CJK TC",
        "Noto Sans CJK SC",
        "Arial Unicode MS",
    ]
    plt.rcParams["font.sans-serif"] = candidates
    plt.rcParams["axes.unicode_minus"] = False


set_chinese_font()
print("HEIC／HEIF 支援：", "可用" if HEIC_AVAILABLE else "未安裝 pillow-heif")


HEIC／HEIF 支援： 未安裝 pillow-heif


## 2. 參數設定

`FACE_CROP_SCALE = 1.15` 較 FF++ 使用的 1.30 更緊。這批照片為一般拍攝構圖，人臉在畫面中的相對比例與 FF++ 影片不同，過大的外擴會納入過多背景。

In [3]:
# 原始手機照片：底下必須直接包含 real 與 fake
MOBILE_RAW_ROOT = Path("./mobile_test_raw")

# 裁切結果會輸出至 test/real 與 test/fake
OUTPUT_ROOT = Path("./dataset_mobile_yolo")

# YOLOv11n-Face 權重
YOLO_MODEL_PATH = Path(r"D:\資料\專題\PY\deepfake\vit(舊版\yolov11n-face.pt")

# YOLO 偵測信心門檻
CONF_THRESHOLD = 0.50

# 1.15 代表在人臉框外額外保留約 15% 範圍
FACE_CROP_SCALE = 1.15

# False：已存在的輸出圖片直接跳過，方便中斷後續跑
# True：重新處理並覆寫同名輸出圖片，但不會刪除整個資料夾
OVERWRITE_OUTPUT = False

# 輸出 JPEG 品質
JPEG_QUALITY = 95

# 正式處理請保持 None；快速測試可改成每類圖片數，例如 5
MAX_IMAGES_PER_CLASS = None

# GPU 可用時使用第 0 張 GPU，否則使用 CPU
DEVICE = 0 if torch.cuda.is_available() else "cpu"

MANIFEST_PATH = OUTPUT_ROOT / "mobile_face_crop_manifest.csv"
FAILED_PATH = OUTPUT_ROOT / "mobile_face_crop_failed.csv"

print("原始資料：", MOBILE_RAW_ROOT.resolve())
print("輸出資料：", (OUTPUT_ROOT / "test").resolve())
print("YOLO 權重：", YOLO_MODEL_PATH.resolve())
print("裝置：", DEVICE)


原始資料： D:\資料\專題\PY\deepfake\新版專題研究\mobile_test_raw
輸出資料： D:\資料\專題\PY\deepfake\新版專題研究\dataset_mobile_yolo\test
YOLO 權重： D:\資料\專題\PY\deepfake\vit(舊版\yolov11n-face.pt
裝置： 0


## 3. 檢查資料夾與圖片

In [4]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff",
    ".heic", ".heif",
}
CLASS_NAMES = ("real", "fake")


if not YOLO_MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"找不到 YOLO 權重：{YOLO_MODEL_PATH.resolve()}\n"
        "請將 yolov11n-face.pt 放在 Notebook 同一資料夾。"
    )

records = []
for class_name in CLASS_NAMES:
    class_dir = MOBILE_RAW_ROOT / class_name
    if not class_dir.is_dir():
        raise FileNotFoundError(
            f"缺少類別資料夾：{class_dir.resolve()}"
        )

    paths = sorted(
        path
        for path in class_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if MAX_IMAGES_PER_CLASS is not None:
        paths = paths[:MAX_IMAGES_PER_CLASS]
    if not paths:
        raise FileNotFoundError(
            f"{class_name} 資料夾中找不到支援的圖片。"
        )

    for source_path in paths:
        relative_path = source_path.relative_to(class_dir)
        output_path = (
            OUTPUT_ROOT
            / "test"
            / class_name
            / relative_path.with_suffix(".jpg")
        )
        records.append({
            "class_name": class_name,
            "source_path": source_path,
            "output_path": output_path,
        })

source_df = pd.DataFrame(records)
display(
    source_df.groupby("class_name", as_index=False)
    .size()
    .rename(columns={"size": "image_count"})
)
display(source_df.head())

if source_df["output_path"].duplicated().any():
    raise ValueError("輸出路徑發生重複，請檢查原始資料夾中的檔名。")


,class_name,image_count
0,fake,150
1,real,150


,class_name,source_path,output_path
0,real,mobile_test_raw\real\LINE_ALBUM_150張Real不同人照片_...,dataset_mobile_yolo\test\real\LINE_ALBUM_150張R...
1,real,mobile_test_raw\real\LINE_ALBUM_150張Real不同人照片_...,dataset_mobile_yolo\test\real\LINE_ALBUM_150張R...
2,real,mobile_test_raw\real\LINE_ALBUM_150張Real不同人照片_...,dataset_mobile_yolo\test\real\LINE_ALBUM_150張R...
3,real,mobile_test_raw\real\LINE_ALBUM_150張Real不同人照片_...,dataset_mobile_yolo\test\real\LINE_ALBUM_150張R...
4,real,mobile_test_raw\real\LINE_ALBUM_150張Real不同人照片_...,dataset_mobile_yolo\test\real\LINE_ALBUM_150張R...


## 4. 裁切函式

In [5]:
def read_phone_image(path):
    """讀取圖片並依手機 EXIF 資訊修正旋轉方向。"""
    suffix = path.suffix.lower()
    if suffix in {".heic", ".heif"} and not HEIC_AVAILABLE:
        raise RuntimeError("HEIC 圖片需要先安裝 pillow-heif")

    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image_rgb = np.asarray(image)
    return cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)


def find_largest_face(detector, image_bgr):
    """傳回面積最大的人臉框、信心分數與偵測人臉數量。"""
    results = detector.predict(
        source=image_bgr,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        return None, None, 0

    boxes = results[0].boxes.xyxy.detach().cpu().numpy()
    confidences = results[0].boxes.conf.detach().cpu().numpy()
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    index = int(np.argmax(areas))
    return boxes[index], float(confidences[index]), len(boxes)


def crop_square_face(image_bgr, box):
    """將人臉框擴張後裁成正方形，超出影像邊界時自動縮回。"""
    image_height, image_width = image_bgr.shape[:2]
    x1, y1, x2, y2 = map(float, box)
    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2

    face_width = max(x2 - x1, 1)
    face_height = max(y2 - y1, 1)
    side = int(round(max(face_width, face_height) * FACE_CROP_SCALE))
    side = max(1, min(side, image_width, image_height))

    left = int(round(center_x - side / 2))
    top = int(round(center_y - side / 2))
    left = min(max(left, 0), image_width - side)
    top = min(max(top, 0), image_height - side)

    face = image_bgr[top:top + side, left:left + side]
    crop_box = (left, top, left + side, top + side)
    return face, crop_box


def write_jpeg_unicode(path, image_bgr):
    """以可支援 Windows 中文路徑的方式儲存 JPEG。"""
    path.parent.mkdir(parents=True, exist_ok=True)
    success, encoded = cv2.imencode(
        ".jpg",
        image_bgr,
        [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
    )
    if not success:
        return False
    encoded.tofile(os.fspath(path))
    return True


## 5. 執行 YOLO 人臉裁切

In [6]:
detector = YOLO(os.fspath(YOLO_MODEL_PATH))
results_records = []

for row in tqdm(
    source_df.itertuples(index=False),
    total=len(source_df),
    desc="YOLO 裁切手機人臉",
):
    source_path = Path(row.source_path)
    output_path = Path(row.output_path)

    result_record = {
        "class_name": row.class_name,
        "source_path": str(source_path.resolve()),
        "output_path": str(output_path.resolve()),
        "status": "",
        "reason": "",
        "detected_face_count": 0,
        "face_confidence": np.nan,
        "original_width": np.nan,
        "original_height": np.nan,
        "crop_left": np.nan,
        "crop_top": np.nan,
        "crop_right": np.nan,
        "crop_bottom": np.nan,
    }

    if output_path.is_file() and not OVERWRITE_OUTPUT:
        result_record["status"] = "skipped_existing"
        results_records.append(result_record)
        continue

    try:
        image_bgr = read_phone_image(source_path)
        image_height, image_width = image_bgr.shape[:2]
        result_record["original_width"] = image_width
        result_record["original_height"] = image_height

        face_box, confidence, face_count = find_largest_face(
            detector, image_bgr
        )
        result_record["detected_face_count"] = face_count

        if face_box is None:
            result_record["status"] = "failed"
            result_record["reason"] = "no_face"
            results_records.append(result_record)
            continue

        face_bgr, crop_box = crop_square_face(image_bgr, face_box)
        if face_bgr.size == 0:
            result_record["status"] = "failed"
            result_record["reason"] = "empty_crop"
            results_records.append(result_record)
            continue

        if not write_jpeg_unicode(output_path, face_bgr):
            result_record["status"] = "failed"
            result_record["reason"] = "write_failed"
            results_records.append(result_record)
            continue

        result_record["status"] = "saved"
        result_record["face_confidence"] = confidence
        (
            result_record["crop_left"],
            result_record["crop_top"],
            result_record["crop_right"],
            result_record["crop_bottom"],
        ) = crop_box

    except (UnidentifiedImageError, OSError, RuntimeError, ValueError) as error:
        result_record["status"] = "failed"
        result_record["reason"] = f"{type(error).__name__}: {error}"
    except Exception as error:
        result_record["status"] = "failed"
        result_record["reason"] = f"unexpected_{type(error).__name__}: {error}"

    results_records.append(result_record)

result_df = pd.DataFrame(results_records)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
result_df.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

failed_df = result_df[result_df["status"] == "failed"].copy()
failed_df.to_csv(FAILED_PATH, index=False, encoding="utf-8-sig")

summary_df = (
    result_df.groupby(["class_name", "status"], as_index=False)
    .size()
    .rename(columns={"size": "image_count"})
)
display(summary_df)

print("處理完成")
print("完整紀錄：", MANIFEST_PATH.resolve())
print("失敗紀錄：", FAILED_PATH.resolve())
print("失敗圖片數：", len(failed_df))


YOLO 裁切手機人臉:   0%|          | 0/300 [00:00<?, ?it/s]

,class_name,status,image_count
0,fake,saved,150
1,real,saved,150


處理完成
完整紀錄： D:\資料\專題\PY\deepfake\新版專題研究\dataset_mobile_yolo\mobile_face_crop_manifest.csv
失敗紀錄： D:\資料\專題\PY\deepfake\新版專題研究\dataset_mobile_yolo\mobile_face_crop_failed.csv
失敗圖片數： 0


## 6. 檢查輸出數量

In [7]:
output_records = []
for class_name in CLASS_NAMES:
    class_dir = OUTPUT_ROOT / "test" / class_name
    paths = sorted(class_dir.rglob("*.jpg")) if class_dir.exists() else []
    output_records.append({
        "class_name": class_name,
        "cropped_image_count": len(paths),
        "folder": str(class_dir.resolve()),
    })

output_summary_df = pd.DataFrame(output_records)
display(output_summary_df)

if (output_summary_df["cropped_image_count"] == 0).any():
    print("警告：至少一個類別沒有成功輸出圖片，請查看失敗紀錄。")
else:
    print("real 與 fake 都已有裁切結果，可交給 09 Notebook 測試。")


,class_name,cropped_image_count,folder
0,real,150,D:\資料\專題\PY\deepfake\新版專題研究\dataset_mobile_yol...
1,fake,150,D:\資料\專題\PY\deepfake\新版專題研究\dataset_mobile_yol...


real 與 fake 都已有裁切結果，可交給 09 Notebook 測試。


## 7. 裁切結果檢查

原本此處為隨機抽樣的裁切結果預覽。由於這批影像包含可識別的個人照片，以及來源為網路的第三方影像，基於肖像權與著作權考量，已於公開版本移除。程式碼保留，可在本機重現。

In [8]:
saved_paths = [
    Path(path)
    for path in result_df.loc[
        result_df["status"] == "saved", "output_path"
    ].tolist()
]

if not saved_paths:
    print("本次沒有新儲存的圖片；若都是 skipped_existing，可直接查看輸出資料夾。")
else:
    preview_paths = saved_paths[:min(8, len(saved_paths))]
    columns = 4
    rows = int(np.ceil(len(preview_paths) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(12, 3 * rows))
    axes = np.atleast_1d(axes).ravel()

    for axis, path in zip(axes, preview_paths):
        with Image.open(path) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(path.parent.name)
        axis.axis("off")

    for axis in axes[len(preview_paths):]:
        axis.axis("off")

    plt.suptitle("YOLO 人臉裁切預覽")
    plt.tight_layout()
    plt.show()


## 輸出與後續步驟

裁切結果輸出至 `dataset_mobile_yolo/test/`，Real 與 Fake 各 150 張。

下一步（09B）將這 300 張重新切分為校準集與正式測試集兩組互斥的資料。**不可**直接以全部 300 張同時做門檻選擇與效能評估——那等於用測試集調整測試條件。